In [1]:
import ee

ee.Authenticate()
ee.Initialize(project='project-0cf410a3-f35f-4911-9c5')

PLACE_NAME = "Tarangire National Park, Tanzania"


Successfully saved authorization token.


In [16]:
import geemap

range = ee.Date('2026-08-24').getRange('day')
dataset = ee.ImageCollection('NASA/GPM_L3/IMERG_V07').filter(
    ee.Filter.date(range)
)

# Select the max precipitation and mask out low precipitation values.
precipitation = dataset.select('precipitation').max()
mask = precipitation.gt(0.5)
precipitation = precipitation.updateMask(mask)

palette = [
  '000096','0064ff', '00b4ff', '33db80', '9beb4a',
  'ffeb00', 'ffb300', 'ff6400', 'eb1e00', 'af0000'
]
precipitation_vis = {'min': 0, 'max': 15, 'palette': palette}

m = geemap.Map()
m.add_layer(precipitation, precipitation_vis, 'Precipitation (mm/hr)')
m.set_center(-76, 33, 3)
m

Map(center=[33, -76], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright', tra…

In [2]:
from park_road_network import ParkRoadNetwork
from imerg_provider import IMERGProvider

nodes, edges = ParkRoadNetwork(PLACE_NAME).load()
provider = IMERGProvider()
grid = provider.frame_grid(edges.total_bounds)

grid.head()


,pixel_id,geometry
0,0,"POLYGON ((35.9 -4.6, 35.9 -4.5, 35.8 -4.5, 35...."
1,1,"POLYGON ((35.9 -4.5, 35.9 -4.4, 35.8 -4.4, 35...."
2,2,"POLYGON ((35.9 -4.4, 35.9 -4.3, 35.8 -4.3, 35...."
3,3,"POLYGON ((35.9 -4.3, 35.9 -4.2, 35.8 -4.2, 35...."
4,4,"POLYGON ((35.9 -4.2, 35.9 -4.1, 35.8 -4.1, 35...."


In [3]:
result, ic = provider.get_accumulated_precipitation(
    grid,
    start_date="2026-08-20",
    end_date="2026-08-24"
)
print(ic.size().getInfo())  # sanity check image count
result[["pixel_id", "rain_mm"]].sort_values("rain_mm", ascending=False).head()

5


,pixel_id,rain_mm
0,0,0
1,1,0
2,2,0
3,3,0
4,4,0


In [5]:
result, ic = provider.get_decayed_precipitation(
    grid,
    date="2026-08-24",
    lookback_days=5
)
result[["pixel_id", "rain_mm"]].describe()

,pixel_id,rain_mm
count,66.000000,66.0
mean,32.500000,0.0
std,19.196354,0.0
min,0.000000,0.0
25%,16.250000,0.0
50%,32.500000,0.0
75%,48.750000,0.0
max,65.000000,0.0


In [6]:
from datetime import date
from imerg_provider import IMERGProvider
from edge_risk_enricher import EdgeRiskEnricher

import folium

def create_risk_map_imerg(lookback_days: int, date: date):

    nodes, edges = ParkRoadNetwork(PLACE_NAME).load()
    rain_provider = IMERGProvider()

    pixel_grid = rain_provider.frame_grid(edges.total_bounds)

    rain_pixels, image_collection = rain_provider.get_decayed_precipitation(
        pixel_grid,
        date=date.strftime("%Y-%m-%d"),
        lookback_days=lookback_days
    )

    enricher = EdgeRiskEnricher()
    edges_enriched = enricher.enrich(edges, rain_pixels)

    center_lat, center_lon = nodes["y"].mean(), nodes["x"].mean()
    m = folium.Map(location=[center_lat, center_lon], zoom_start=10)

    SURFACE_RISK_COLORS = {
        (0.0, 0.25): "#2ecc71",
        (0.25, 0.5): "#f39c12",
        (0.5, 0.75): "#e74c3c",
        (0.75, 1.01): "#7b241c",
    }

    def get_risk_color(surface_risk: float) -> str:
        for (low, high), color in SURFACE_RISK_COLORS.items():
            if low <= surface_risk < high:
                return color
        return "#7b241c"

    for _, row in edges_enriched.iterrows():
        coords = [(lat, lon) for lon, lat in row["geometry"].coords]
        road_type = row["road_type"]
        rain_mm = row["rain_mm"]
        travel_time = row["travel_time_m"]

        surface_risk = row["surface_risk"]

        folium.PolyLine(
            locations=coords,
            color=get_risk_color(surface_risk),
            weight=3,
            opacity=0.8,
            popup=(
                f"highway: {road_type}<br>"
                f"travel_time: {travel_time}<br>"
                f"rain_mm(accumulated over x (start - end) days): {rain_mm:.1f}<br>"
                f"surface_risk: {surface_risk:.2f}"
            ),
        ).add_to(m)

    return m

In [7]:
~~~~import ipywidgets as widgets
from IPython.display import display, clear_output
from datetime import date, datetime

date_picker_imerg = widgets.DatePicker(
    description="Date:",
    value=date.today(),  # IMERG's use case is recent/current dates, unlike CHIRPS Final
)

lookback_days_input_imerg = widgets.IntText(
    value=3,
    description="Lookback days:",
    disabled=False,
)

load_button_imerg = widgets.Button(
    description="Load Map (IMERG)",
    button_style="primary",
)

output_imerg = widgets.Output()


def on_render_clicked_imerg(_):
    d = date_picker_imerg.value
    lookback_days = lookback_days_input_imerg.value

    with output_imerg:
        clear_output(wait=True)
        try:
            m = create_risk_map_imerg(date=d, lookback_days=lookback_days)
            display(m)
        except ValueError as e:
            print(f"IMERG coverage error: {e}")


load_button_imerg.on_click(on_render_clicked_imerg)

display(
    widgets.HBox([date_picker_imerg, lookback_days_input_imerg, load_button_imerg]),
    output_imerg,
)

Output()